# 05 — EDA Financeira Forense: Valorajuizado + Target Banco Ganhou

Este notebook refaz a análise considerando as alterações recentes:

- filtro para processos iniciados a partir de 2015;
- apenas `Tipoprocesso == PASSIVO`;
- apenas carteiras `MASSIFICADO` e `MASSIFICADO - REVISIONAIS`;
- apenas `Situacao == ENCERRADO`;
- apenas `Valorajuizado > 0`;
- criação do target `target_banco_ganhou` a partir de `Motivo_Encerramento`;
- remoção dos casos sem target.

A análise agora deixa de ser apenas uma análise de exposição e passa a ser uma análise de **perda financeira histórica**: onde o banco ganhou, onde perdeu e onde o valor ajuizado ficou concentrado.

## Target

- `target_banco_ganhou = 1`: banco ganhou / êxito.
- `target_banco_ganhou = 0`: banco perdeu / condenação.
- `NaN`: desfecho indefinido ou acordo não classificado como perda.

Por padrão, acordos genéricos não são tratados como perda (`ACORDO_COMO_PERDA = False`). Acordos pós-sentença ou em execução são tratados como condenação, seguindo a regra jurídica definida.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)

REPORTS_DIR = Path("../outputs/reports")
PROCESSED_DIR = Path("../data/processed")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

VALUE_COL = "Valorajuizado"
print("Setup concluído")

## 2. Validar ou carregar `df_joined`

Este notebook assume que `df_joined` já existe em memória, vindo do join entre Benner e DeepLegal Processos.

Se preferir carregar de arquivo, descomente uma das linhas abaixo.

In [ ]:
# Opcional:
# df_joined = pd.read_parquet("../data/processed/df_joined_benner_deeplegal.parquet")
# df_joined = pd.read_csv("../data/processed/df_joined_benner_deeplegal.csv")

try:
    df_joined
    print("df_joined encontrado em memória.")
    print("Shape original:", df_joined.shape)
except NameError:
    raise NameError("df_joined não existe em memória. Rode antes o notebook de preparação ou carregue o arquivo parquet/csv.")

df_joined.head()

## 3. Funções utilitárias e regras de target

In [ ]:
def norm_txt(s) -> str:
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    s = str(s).strip().upper()
    s = unicodedata.normalize("NFKD", s)
    return "".join(c for c in s if not unicodedata.combining(c))

def normalize_text(x):
    if pd.isna(x):
        return None
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKD", x)
    x = "".join(c for c in x if not unicodedata.combining(c))
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x if x not in {"", "null", "nan", "none", "na", "n/a", "-"} else None

def save_report(df, filename):
    path = REPORTS_DIR / filename
    df.to_csv(path, index=False)
    print(f"Salvo: {path}")

def existing_cols(df, cols):
    return [c for c in cols if c in df.columns]

def sanitize_filename_col(col):
    return re.sub(r"[^a-zA-Z0-9_]", "_", str(col))

def parse_sim_nao(x):
    x = normalize_text(x)
    if x in ["sim", "s", "yes", "true", "1"]:
        return 1
    if x in ["nao", "não", "n", "no", "false", "0"]:
        return 0
    return np.nan

def safe_divide(a, b):
    return np.where((b == 0) | pd.isna(b), np.nan, a / b)

# True = demais acordos contam como perda (0)
# False = demais acordos ficam como indefinidos / NaN
ACORDO_COMO_PERDA = False

ACORDO_CONDENACAO_EXATOS = {
    "ACORDO POS SENTENCA",
    "ACORDO EM EXECUCAO",
}

def _motivo_acordo_condenacao(mot: str) -> bool:
    if "ACORDO" not in mot:
        return False
    if mot in ACORDO_CONDENACAO_EXATOS:
        return True
    if "POS SENTENCA" in mot and "ACORDO" in mot:
        return True
    if "EM EXECUCAO" in mot and "ACORDO" in mot:
        return True
    return False

def classificar_desfecho_banco(situacao, motivo) -> str:
    sit = norm_txt(situacao)
    mot = norm_txt(motivo)

    if sit != "ENCERRADO":
        return "indefinido_em_curso"
    if not mot:
        return "indefinido_sem_motivo"
    if "LIMPEZA DE BASE" in mot or "BAIXA DE JURIDICO" in mot:
        return "indefinido_admin"

    # Acordos que o jurídico orientou tratar como condenação
    if _motivo_acordo_condenacao(mot):
        return "condenacao"

    # Perda / condenação judicial
    if "PARCIAL" in mot and "PROCEDENTE" in mot:
        return "condenacao"
    if "IMPROCEDENTE" in mot or "IMPROCEDENCIA" in mot:
        return "exito"
    if "PROCEDENTE" in mot or "PROCEDENCIA" in mot:
        return "condenacao"

    # Êxito: extinção / sem mérito / desistência / prescrição etc.
    exito_kw = (
        "EXTINCAO", "SEM JULGAMENTO DE MERITO", "DESISTENCIA",
        "RENUNCIA", "PRESCRIC", "DECADENCIA", "ARQUIVADO"
    )
    if any(k in mot for k in exito_kw):
        return "exito"

    # Demais acordos
    if "ACORDO" in mot:
        return "acordo"

    return "indefinido_outros"

def target_banco_ganhou(situacao, motivo):
    cat = classificar_desfecho_banco(situacao, motivo)
    if cat == "exito":
        return 1
    if cat == "condenacao":
        return 0
    if cat == "acordo":
        return 0 if ACORDO_COMO_PERDA else np.nan
    return np.nan

## 4. Aplicar filtro atualizado

O filtro abaixo reproduz o seu novo recorte operacional: processos passivos, massificados/revisionais, encerrados e com valor positivo.

In [ ]:
df_eda = df_joined.copy()

required_cols = ["Datainicial", "Tipoprocesso", "Carteira", "Situacao", VALUE_COL, "Motivo_Encerramento"]
missing = [c for c in required_cols if c not in df_eda.columns]
if missing:
    raise KeyError(f"Colunas obrigatórias ausentes: {missing}")

df_eda["Datainicial"] = pd.to_datetime(df_eda["Datainicial"], errors="coerce")
df_eda[VALUE_COL] = pd.to_numeric(df_eda[VALUE_COL], errors="coerce")

df_eda = df_eda[
    (df_eda["Datainicial"] >= "2015-01-01") &
    (df_eda["Tipoprocesso"] == "PASSIVO") &
    (df_eda["Carteira"].isin(["MASSIFICADO", "MASSIFICADO - REVISIONAIS"])) &
    (df_eda["Situacao"] == "ENCERRADO") &
    (df_eda[VALUE_COL] > 0)
].copy()

df_eda["valor_ajuizado"] = df_eda[VALUE_COL]

print("Shape após filtro:", df_eda.shape)
df_eda.head()

## 5. Criar `desfecho_categoria` e `target_banco_ganhou`

In [ ]:
df_eda["desfecho_categoria"] = df_eda.apply(
    lambda r: classificar_desfecho_banco(r["Situacao"], r["Motivo_Encerramento"]),
    axis=1,
)

df_eda["target_banco_ganhou"] = df_eda.apply(
    lambda r: target_banco_ganhou(r["Situacao"], r["Motivo_Encerramento"]),
    axis=1,
)

target_audit_before_drop = (
    df_eda
    .groupby("desfecho_categoria", dropna=False)
    .agg(
        qtd_processos=("valor_ajuizado", "size"),
        valor_total=("valor_ajuizado", "sum"),
        valor_medio=("valor_ajuizado", "mean"),
        target_nao_nulo=("target_banco_ganhou", "count")
    )
    .reset_index()
    .sort_values("qtd_processos", ascending=False)
)

save_report(target_audit_before_drop, "target_audit_before_drop.csv")
target_audit_before_drop

## 6. Remover linhas sem target

Após esta etapa, a base deve ficar próxima dos **46.628 processos** que você observou.

In [ ]:
df_model_eda = df_eda.dropna(subset=["target_banco_ganhou"]).copy()
df_model_eda["target_banco_ganhou"] = df_model_eda["target_banco_ganhou"].astype(int)
df_model_eda["target_banco_perdeu"] = 1 - df_model_eda["target_banco_ganhou"]

print("Shape antes de remover target nulo:", df_eda.shape)
print("Shape após remover target nulo:", df_model_eda.shape)

target_distribution = (
    df_model_eda
    .groupby("target_banco_ganhou")
    .agg(
        qtd_processos=("valor_ajuizado", "size"),
        valor_total=("valor_ajuizado", "sum"),
        valor_medio=("valor_ajuizado", "mean"),
        valor_mediano=("valor_ajuizado", "median")
    )
    .reset_index()
)
target_distribution["classe"] = target_distribution["target_banco_ganhou"].map({1: "banco_ganhou", 0: "banco_perdeu"})
target_distribution["perc_processos"] = target_distribution["qtd_processos"] / target_distribution["qtd_processos"].sum()
target_distribution["perc_valor_total"] = target_distribution["valor_total"] / target_distribution["valor_total"].sum()

save_report(target_distribution, "target_distribution_banco_ganhou.csv")
target_distribution

## 7. Diagnóstico financeiro geral e por target

Aqui verificamos se os processos em que o banco perdeu possuem valor médio, mediano ou cauda financeira maior do que os processos em que o banco ganhou.

In [ ]:
valor_summary_target = df_model_eda["valor_ajuizado"].describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).to_frame("valor_ajuizado")

save_report(valor_summary_target.reset_index(), "valorajuizado_summary_target_definido.csv")
display(valor_summary_target)

valor_by_target = (
    df_model_eda
    .groupby("target_banco_ganhou")
    .agg(
        qtd_processos=("valor_ajuizado", "size"),
        valor_total=("valor_ajuizado", "sum"),
        valor_medio=("valor_ajuizado", "mean"),
        valor_mediano=("valor_ajuizado", "median"),
        valor_p75=("valor_ajuizado", lambda x: x.quantile(0.75)),
        valor_p90=("valor_ajuizado", lambda x: x.quantile(0.90)),
        valor_p95=("valor_ajuizado", lambda x: x.quantile(0.95)),
        valor_p99=("valor_ajuizado", lambda x: x.quantile(0.99)),
    )
    .reset_index()
)
valor_by_target["classe"] = valor_by_target["target_banco_ganhou"].map({1: "banco_ganhou", 0: "banco_perdeu"})
valor_by_target["perc_processos"] = valor_by_target["qtd_processos"] / valor_by_target["qtd_processos"].sum()
valor_by_target["perc_valor_total"] = valor_by_target["valor_total"] / valor_by_target["valor_total"].sum()

save_report(valor_by_target, "valorajuizado_by_target_banco_ganhou.csv")
valor_by_target

## 8. Pareto financeiro por target

In [ ]:
def pareto_value_analysis(df, value_col="valor_ajuizado", label="geral"):
    temp = df[df[value_col].notna() & (df[value_col] > 0)].copy()
    temp = temp.sort_values(value_col, ascending=False)
    total = temp[value_col].sum()
    if len(temp) == 0 or total == 0:
        return pd.DataFrame()
    temp["rank_percentual"] = np.arange(1, len(temp) + 1) / len(temp)
    rows = []
    for pct in [0.01, 0.05, 0.10, 0.20, 0.30]:
        mask = temp["rank_percentual"] <= pct
        rows.append({
            "grupo": label,
            "top_percentual_processos": pct,
            "qtd_processos": int(mask.sum()),
            "valor_total_concentrado": temp.loc[mask, value_col].sum(),
            "perc_valor_total_concentrado": temp.loc[mask, value_col].sum() / total
        })
    return pd.DataFrame(rows)

pareto_target = pd.concat([
    pareto_value_analysis(df_model_eda, label="todos"),
    pareto_value_analysis(df_model_eda[df_model_eda["target_banco_ganhou"] == 1], label="banco_ganhou"),
    pareto_value_analysis(df_model_eda[df_model_eda["target_banco_ganhou"] == 0], label="banco_perdeu"),
], ignore_index=True)

save_report(pareto_target, "pareto_valorajuizado_by_target.csv")
pareto_target

## 9. Faixas de valor e taxa de vitória

In [ ]:
bins = [-np.inf, 1_000, 5_000, 10_000, 25_000, 50_000, 100_000, 250_000, 500_000, 1_000_000, np.inf]
labels = ["01_ate_1k", "02_1k_5k", "03_5k_10k", "04_10k_25k", "05_25k_50k", "06_50k_100k", "07_100k_250k", "08_250k_500k", "09_500k_1mm", "10_acima_1mm"]

df_model_eda["faixa_valor_ajuizado"] = pd.cut(df_model_eda["valor_ajuizado"], bins=bins, labels=labels, include_lowest=True).astype(str)

df_model_eda["valor_perdido"] = np.where(df_model_eda["target_banco_ganhou"] == 0, df_model_eda["valor_ajuizado"], 0)
df_model_eda["valor_ganho"] = np.where(df_model_eda["target_banco_ganhou"] == 1, df_model_eda["valor_ajuizado"], 0)

faixa_target_report = (
    df_model_eda
    .groupby("faixa_valor_ajuizado", dropna=False)
    .agg(
        qtd_processos=("valor_ajuizado", "size"),
        valor_total=("valor_ajuizado", "sum"),
        valor_medio=("valor_ajuizado", "mean"),
        taxa_banco_ganhou=("target_banco_ganhou", "mean"),
        taxa_banco_perdeu=("target_banco_perdeu", "mean"),
        valor_perdido_total=("valor_perdido", "sum"),
        valor_ganho_total=("valor_ganho", "sum"),
    )
    .reset_index()
)
faixa_target_report["perc_processos"] = faixa_target_report["qtd_processos"] / faixa_target_report["qtd_processos"].sum()
faixa_target_report["perc_valor_total"] = faixa_target_report["valor_total"] / faixa_target_report["valor_total"].sum()
faixa_target_report["share_valor_perdido"] = faixa_target_report["valor_perdido_total"] / faixa_target_report["valor_perdido_total"].sum()

save_report(faixa_target_report, "faixa_valorajuizado_taxa_vitoria.csv")
faixa_target_report

## 10. Função central: valor, perda e target por grupo

In [ ]:
def report_by_group_with_target(df, group_col, value_col="valor_ajuizado", target_col="target_banco_ganhou", min_count=30):
    if group_col not in df.columns:
        print(f"Coluna não encontrada: {group_col}")
        return None
    temp = df.copy()
    temp["valor_perdido"] = np.where(temp[target_col] == 0, temp[value_col], 0)
    temp["valor_ganho"] = np.where(temp[target_col] == 1, temp[value_col], 0)
    out = (
        temp.groupby(group_col, dropna=False)
        .agg(
            qtd_processos=(value_col, "size"),
            valor_total=(value_col, "sum"),
            valor_medio=(value_col, "mean"),
            valor_mediano=(value_col, "median"),
            valor_p90=(value_col, lambda x: x.quantile(0.90)),
            valor_p95=(value_col, lambda x: x.quantile(0.95)),
            taxa_banco_ganhou=(target_col, "mean"),
            valor_perdido_total=("valor_perdido", "sum"),
            valor_ganho_total=("valor_ganho", "sum"),
            qtd_perdas=("target_banco_perdeu", "sum"),
            qtd_ganhos=(target_col, "sum"),
        )
        .reset_index()
    )
    out = out[out["qtd_processos"] >= min_count].copy()
    out["taxa_banco_perdeu"] = 1 - out["taxa_banco_ganhou"]
    out["perc_processos"] = out["qtd_processos"] / out["qtd_processos"].sum()
    out["share_valor_total"] = out["valor_total"] / out["valor_total"].sum()
    out["share_valor_perdido"] = out["valor_perdido_total"] / out["valor_perdido_total"].sum()
    out["valor_perdido_medio_por_processo"] = out["valor_perdido_total"] / out["qtd_processos"]
    out["valor_perdido_medio_por_perda"] = out["valor_perdido_total"] / out["qtd_perdas"].replace(0, np.nan)
    return out.sort_values(["valor_perdido_total", "valor_total"], ascending=False)

## 11. Relatórios por variáveis relevantes

In [ ]:
candidate_group_cols = [
    # Benner
    "Carteira", "Nm_Produto", "Nm_Acao", "Nm_Assunto", "Tipoprocesso", "Uf", "Comarca",
    "Fasedoprocesso", "Situacao", "Motivo_Encerramento", "desfecho_categoria", "Estimativa_De_Perda",
    "Passiveldeacordo", "Processorelevante", "Credenciado", "Adv_Interno", "Advogadoadverso", "Nm_Empresa",
    # DeepLegal Processos
    "classe_texto", "area_texto", "assunto_normalizado_texto", "assunto_texto", "fase_processual_texto",
    "status_texto", "status_tipo", "cidade", "uf", "vara_texto", "orgao_julgador_texto", "juiz_normalizado_texto",
    "tipo_de_processo_texto", "tipo_de_justica_texto", "tipo_de_requerente_texto", "tipo_de_requerido_texto",
    "resultado_final_processo_text", "resultadojulgamento_tipo", "sentenca_tipo", "resultado_do_recurso_texto", "resultado_exec_texto",
]

available_group_cols = [c for c in candidate_group_cols if c in df_model_eda.columns]
target_group_reports = {}

for col in available_group_cols:
    print("\n" + "=" * 120)
    print(col)
    report = report_by_group_with_target(df_model_eda, group_col=col, min_count=30)
    if report is not None and len(report) > 0:
        target_group_reports[col] = report
        save_report(report, f"target_valorajuizado_by_{sanitize_filename_col(col)}.csv")
        display(report.head(20))

## 12. Priorização por assunto

In [ ]:
if "Nm_Assunto" in target_group_reports:
    assunto_priority = target_group_reports["Nm_Assunto"].copy()
    volume_median = assunto_priority["qtd_processos"].median()
    valor_perdido_median = assunto_priority["valor_perdido_total"].median()
    taxa_perda_median = assunto_priority["taxa_banco_perdeu"].median()

    assunto_priority["volume_alto"] = assunto_priority["qtd_processos"] >= volume_median
    assunto_priority["valor_perdido_alto"] = assunto_priority["valor_perdido_total"] >= valor_perdido_median
    assunto_priority["taxa_perda_alta"] = assunto_priority["taxa_banco_perdeu"] >= taxa_perda_median

    def classify_priority(row):
        if row["valor_perdido_alto"] and row["volume_alto"] and row["taxa_perda_alta"]:
            return "prioridade_1_atacar_agora"
        if row["valor_perdido_alto"] and row["volume_alto"]:
            return "prioridade_2_alta_perda_em_massa"
        if row["valor_perdido_alto"] and not row["volume_alto"]:
            return "prioridade_3_casos_alto_valor"
        if row["volume_alto"] and row["taxa_perda_alta"]:
            return "prioridade_4_revisar_tese_ou_operacao"
        return "monitorar"

    def recommend_action(row):
        p = row["prioridade"]
        if p == "prioridade_1_atacar_agora":
            return "Atacar agora: revisar tese, documentação, causa raiz e estratégia de acordo"
        if p == "prioridade_2_alta_perda_em_massa":
            return "Criar esteira de massa: padronização de defesa, automação e negociação em lote"
        if p == "prioridade_3_casos_alto_valor":
            return "Governança executiva e acompanhamento individual dos casos de alto valor"
        if p == "prioridade_4_revisar_tese_ou_operacao":
            return "Investigar tese jurídica, prova/documentação e origem operacional da recorrência"
        return "Monitorar"

    assunto_priority["prioridade"] = assunto_priority.apply(classify_priority, axis=1)
    assunto_priority["acao_recomendada"] = assunto_priority.apply(recommend_action, axis=1)
    assunto_priority = assunto_priority.sort_values("valor_perdido_total", ascending=False)
    save_report(assunto_priority, "prioridade_financeira_target_by_nm_assunto.csv")
    display(assunto_priority.head(50))
else:
    print("Nm_Assunto não disponível.")

## 13. Produto × Ação × Assunto e Ação × Assunto

In [ ]:
if all(c in df_model_eda.columns for c in ["Nm_Produto", "Nm_Acao", "Nm_Assunto"]):
    df_model_eda["produto_acao_assunto"] = (
        df_model_eda["Nm_Produto"].astype(str).fillna("NAO_INFORMADO") + " | " +
        df_model_eda["Nm_Acao"].astype(str).fillna("NAO_INFORMADO") + " | " +
        df_model_eda["Nm_Assunto"].astype(str).fillna("NAO_INFORMADO")
    )
    produto_acao_assunto_report = report_by_group_with_target(df_model_eda, "produto_acao_assunto", min_count=20)
    save_report(produto_acao_assunto_report, "target_valorajuizado_by_produto_acao_assunto.csv")
    display(produto_acao_assunto_report.head(50))

if all(c in df_model_eda.columns for c in ["Nm_Acao", "Nm_Assunto"]):
    df_model_eda["acao_assunto"] = (
        df_model_eda["Nm_Acao"].astype(str).fillna("NAO_INFORMADO") + " | " +
        df_model_eda["Nm_Assunto"].astype(str).fillna("NAO_INFORMADO")
    )
    acao_assunto_report = report_by_group_with_target(df_model_eda, "acao_assunto", min_count=20)
    save_report(acao_assunto_report, "target_valorajuizado_by_acao_assunto.csv")
    display(acao_assunto_report.head(50))

## 14. Fase, acordo, território e responsáveis

In [ ]:
for col in existing_cols(df_model_eda, ["Fasedoprocesso", "fase_processual_texto", "status_texto"]):
    report = report_by_group_with_target(df_model_eda, col, min_count=30)
    save_report(report, f"target_valorajuizado_by_{sanitize_filename_col(col)}.csv")
    print("\nFase/status:", col)
    display(report.head(30))

if "Passiveldeacordo" in df_model_eda.columns:
    df_model_eda["flag_passivel_acordo"] = df_model_eda["Passiveldeacordo"].apply(parse_sim_nao)
    acordo_report = report_by_group_with_target(df_model_eda, "Passiveldeacordo", min_count=1)
    save_report(acordo_report, "target_valorajuizado_by_passiveldeacordo.csv")
    display(acordo_report)

for col in existing_cols(df_model_eda, ["Uf", "Comarca", "uf", "cidade", "vara_texto", "orgao_julgador_texto"]):
    report = report_by_group_with_target(df_model_eda, col, min_count=30)
    save_report(report, f"target_valorajuizado_by_{sanitize_filename_col(col)}.csv")
    print("\nTerritório:", col)
    display(report.head(20))

for col in existing_cols(df_model_eda, ["Credenciado", "Adv_Interno", "Advogadoadverso"]):
    report = report_by_group_with_target(df_model_eda, col, min_count=30)
    save_report(report, f"target_valorajuizado_by_{sanitize_filename_col(col)}.csv")
    print("\nResponsável:", col)
    display(report.head(20))

## 15. Matriz volume × valor perdido × taxa de perda

In [ ]:
def add_priority_quadrants(report):
    out = report.copy()
    volume_median = out["qtd_processos"].median()
    lost_value_median = out["valor_perdido_total"].median()
    loss_rate_median = out["taxa_banco_perdeu"].median()
    out["quadrante_volume_perda"] = np.select(
        [
            (out["qtd_processos"] >= volume_median) & (out["valor_perdido_total"] >= lost_value_median) & (out["taxa_banco_perdeu"] >= loss_rate_median),
            (out["qtd_processos"] >= volume_median) & (out["valor_perdido_total"] >= lost_value_median),
            (out["qtd_processos"] < volume_median) & (out["valor_perdido_total"] >= lost_value_median),
            (out["qtd_processos"] >= volume_median) & (out["taxa_banco_perdeu"] >= loss_rate_median),
        ],
        [
            "alto_volume_alta_perda_alta_taxa",
            "alto_volume_alta_perda",
            "baixo_volume_alta_perda",
            "alto_volume_alta_taxa_perda",
        ],
        default="monitorar"
    )
    return out

quadrant_reports = {}
for col in ["Nm_Assunto", "Nm_Produto", "Nm_Acao", "produto_acao_assunto", "acao_assunto"]:
    if col in df_model_eda.columns:
        report = report_by_group_with_target(df_model_eda, col, min_count=20)
        if report is not None and len(report) > 0:
            quad = add_priority_quadrants(report)
            quadrant_reports[col] = quad
            save_report(quad, f"quadrante_volume_perda_by_{sanitize_filename_col(col)}.csv")
            print("\n", col)
            display(quad.head(30))

## 16. Análise temporal e duração até encerramento

In [ ]:
df_model_eda["ano_inicio"] = pd.to_datetime(df_model_eda["Datainicial"], errors="coerce").dt.year

temporal_report = (
    df_model_eda
    .groupby("ano_inicio", dropna=False)
    .agg(
        qtd_processos=("valor_ajuizado", "size"),
        valor_total=("valor_ajuizado", "sum"),
        valor_medio=("valor_ajuizado", "mean"),
        taxa_banco_ganhou=("target_banco_ganhou", "mean"),
        taxa_banco_perdeu=("target_banco_perdeu", "mean"),
        valor_perdido_total=("valor_perdido", "sum"),
        valor_ganho_total=("valor_ganho", "sum"),
    )
    .reset_index()
    .sort_values("ano_inicio")
)
save_report(temporal_report, "temporal_valor_target_by_ano_inicio.csv")
display(temporal_report)

if "Dataencerramento" in df_model_eda.columns:
    df_model_eda["Dataencerramento"] = pd.to_datetime(df_model_eda["Dataencerramento"], errors="coerce")
    df_model_eda["duracao_ate_encerramento_dias"] = (df_model_eda["Dataencerramento"] - df_model_eda["Datainicial"]).dt.days
    df_model_eda["faixa_duracao_encerramento"] = pd.cut(
        df_model_eda["duracao_ate_encerramento_dias"],
        bins=[-np.inf, 180, 365, 730, 1095, 1825, np.inf],
        labels=["ate_6_meses", "6m_1ano", "1_2_anos", "2_3_anos", "3_5_anos", "acima_5_anos"]
    ).astype(str)
    duration_report = report_by_group_with_target(df_model_eda, "faixa_duracao_encerramento", min_count=1)
    save_report(duration_report, "target_valorajuizado_by_faixa_duracao_encerramento.csv")
    display(duration_report)

## 17. Divergência Benner × DeepLegal, se `valor_valor` existir

In [ ]:
if "valor_valor" in df_model_eda.columns:
    df_model_eda["valor_deeplegal"] = pd.to_numeric(df_model_eda["valor_valor"], errors="coerce")
    df_model_eda["abs_diff_valor_benner_dl"] = (df_model_eda["valor_ajuizado"] - df_model_eda["valor_deeplegal"]).abs()
    df_model_eda["diff_relativa_benner_dl"] = safe_divide(
        df_model_eda["abs_diff_valor_benner_dl"],
        df_model_eda[["valor_ajuizado", "valor_deeplegal"]].max(axis=1)
    )
    df_model_eda["flag_divergencia_alta_benner_dl"] = (
        (df_model_eda["abs_diff_valor_benner_dl"] >= 10_000) &
        (df_model_eda["diff_relativa_benner_dl"] >= 0.50)
    ).astype(int)
    divergencia_summary = pd.DataFrame([{
        "qtd_com_ambos_valores": int((df_model_eda["valor_ajuizado"].notna() & df_model_eda["valor_deeplegal"].notna()).sum()),
        "correlacao_benner_deeplegal": df_model_eda[["valor_ajuizado", "valor_deeplegal"]].corr().iloc[0, 1],
        "qtd_divergencia_alta": int(df_model_eda["flag_divergencia_alta_benner_dl"].sum()),
        "perc_divergencia_alta": float(df_model_eda["flag_divergencia_alta_benner_dl"].mean()),
        "media_abs_diff": float(df_model_eda["abs_diff_valor_benner_dl"].mean()),
        "mediana_abs_diff": float(df_model_eda["abs_diff_valor_benner_dl"].median()),
        "p95_abs_diff": float(df_model_eda["abs_diff_valor_benner_dl"].quantile(0.95)),
    }])
    save_report(divergencia_summary, "divergencia_valor_benner_deeplegal_encerrados_summary.csv")
    display(divergencia_summary)
else:
    print("Coluna valor_valor da DeepLegal não encontrada.")

## 18. Curva de ganho financeiro por valor perdido

In [ ]:
df_gain_loss = df_model_eda.sort_values("valor_perdido", ascending=False).copy()
df_gain_loss["rank"] = np.arange(1, len(df_gain_loss) + 1)
df_gain_loss["perc_processos"] = df_gain_loss["rank"] / len(df_gain_loss)
df_gain_loss["valor_perdido_acumulado"] = df_gain_loss["valor_perdido"].cumsum()
df_gain_loss["perc_valor_perdido_acumulado"] = df_gain_loss["valor_perdido_acumulado"] / df_gain_loss["valor_perdido"].sum()

gain_loss_summary = []
for pct in [0.01, 0.05, 0.10, 0.20, 0.30]:
    temp = df_gain_loss[df_gain_loss["perc_processos"] <= pct]
    gain_loss_summary.append({
        "top_percentual_processos": pct,
        "qtd_processos": len(temp),
        "valor_perdido_capturado": temp["valor_perdido"].sum(),
        "perc_valor_perdido_capturado": temp["valor_perdido"].sum() / df_gain_loss["valor_perdido"].sum(),
        "valor_total_capturado": temp["valor_ajuizado"].sum(),
        "perc_valor_total_capturado": temp["valor_ajuizado"].sum() / df_gain_loss["valor_ajuizado"].sum(),
    })

gain_loss_summary = pd.DataFrame(gain_loss_summary)
save_report(gain_loss_summary, "curva_ganho_financeiro_valor_perdido_summary.csv")
save_report(df_gain_loss, "curva_ganho_financeiro_valor_perdido_detalhada.csv")
gain_loss_summary

## 19. Temas críticos retrospectivos

In [ ]:
quick_win_group_col = "produto_acao_assunto" if "produto_acao_assunto" in df_model_eda.columns else "Nm_Assunto"

quick_win_report = report_by_group_with_target(df_model_eda, group_col=quick_win_group_col, min_count=20)
if quick_win_report is not None and len(quick_win_report) > 0:
    p75_valor_perdido = quick_win_report["valor_perdido_total"].quantile(0.75)
    p75_taxa_perda = quick_win_report["taxa_banco_perdeu"].quantile(0.75)
    quick_win_report["flag_tema_critico_retrospectivo"] = (
        (quick_win_report["valor_perdido_total"] >= p75_valor_perdido) &
        (quick_win_report["taxa_banco_perdeu"] >= p75_taxa_perda)
    ).astype(int)
    quick_win_report = quick_win_report.sort_values(["flag_tema_critico_retrospectivo", "valor_perdido_total"], ascending=False)
    save_report(quick_win_report, "temas_criticos_retrospectivos_perda_financeira.csv")
    display(quick_win_report.head(50))

## 20. Ranking final de processos encerrados por valor perdido

In [ ]:
ranking_cols = existing_cols(df_model_eda, [
    "Numerodistribuicao", "Identificador", "Carteira", "Nm_Produto", "Nm_Acao", "Nm_Assunto", "Tipoprocesso",
    "Uf", "Comarca", "Fasedoprocesso", "Situacao", "Motivo_Encerramento", "desfecho_categoria", "target_banco_ganhou",
    "valor_ajuizado", "valor_perdido", "valor_ganho", "faixa_valor_ajuizado", "Passiveldeacordo", "Processorelevante",
    "Credenciado", "Adv_Interno", "classe_texto", "assunto_normalizado_texto", "fase_processual_texto", "status_texto",
    "cidade", "uf", "vara_texto", "orgao_julgador_texto"
])

ranking_perdas_processos = df_model_eda[ranking_cols].sort_values(["valor_perdido", "valor_ajuizado"], ascending=False)
save_report(ranking_perdas_processos, "ranking_processos_encerrados_por_valor_perdido.csv")
ranking_perdas_processos.head(100)

## 21. Resumo executivo e salvamento das bases

In [ ]:
executive_summary = pd.DataFrame([{
    "qtd_processos_com_target": len(df_model_eda),
    "qtd_banco_ganhou": int((df_model_eda["target_banco_ganhou"] == 1).sum()),
    "qtd_banco_perdeu": int((df_model_eda["target_banco_ganhou"] == 0).sum()),
    "taxa_banco_ganhou": float(df_model_eda["target_banco_ganhou"].mean()),
    "taxa_banco_perdeu": float(df_model_eda["target_banco_perdeu"].mean()),
    "valor_total_ajuizado": float(df_model_eda["valor_ajuizado"].sum()),
    "valor_total_processos_ganhos": float(df_model_eda.loc[df_model_eda["target_banco_ganhou"] == 1, "valor_ajuizado"].sum()),
    "valor_total_processos_perdidos": float(df_model_eda.loc[df_model_eda["target_banco_ganhou"] == 0, "valor_ajuizado"].sum()),
    "share_valor_perdido": float(df_model_eda.loc[df_model_eda["target_banco_ganhou"] == 0, "valor_ajuizado"].sum() / df_model_eda["valor_ajuizado"].sum()),
    "valor_medio_geral": float(df_model_eda["valor_ajuizado"].mean()),
    "valor_medio_ganho": float(df_model_eda.loc[df_model_eda["target_banco_ganhou"] == 1, "valor_ajuizado"].mean()),
    "valor_medio_perdido": float(df_model_eda.loc[df_model_eda["target_banco_ganhou"] == 0, "valor_ajuizado"].mean()),
    "valor_p95_geral": float(df_model_eda["valor_ajuizado"].quantile(0.95)),
    "valor_p95_perdido": float(df_model_eda.loc[df_model_eda["target_banco_ganhou"] == 0, "valor_ajuizado"].quantile(0.95)),
}])

save_report(executive_summary, "executive_summary_eda_valorajuizado_target.csv")
display(executive_summary)

df_model_eda.to_parquet(PROCESSED_DIR / "abt_eda_valorajuizado_target_banco_ganhou.parquet", index=False)
ranking_perdas_processos.to_parquet(PROCESSED_DIR / "ranking_processos_encerrados_por_valor_perdido.parquet", index=False)

if "assunto_priority" in globals():
    assunto_priority.to_parquet(PROCESSED_DIR / "prioridade_financeira_target_by_nm_assunto.parquet", index=False)

print("ABT salva em:", PROCESSED_DIR / "abt_eda_valorajuizado_target_banco_ganhou.parquet")
print("Ranking salvo em:", PROCESSED_DIR / "ranking_processos_encerrados_por_valor_perdido.parquet")

# Roteiro de explicação para o jurídico

## 1. Mudança de escopo

A análise agora usa apenas processos encerrados, passivos, massificados/revisionais, com data inicial a partir de 2015 e valor ajuizado positivo. Isso permite olhar não só exposição, mas **resultado histórico**.

## 2. O que o target mede

`target_banco_ganhou = 1` significa êxito do banco.  
`target_banco_ganhou = 0` significa perda/condenação.  
Acordos genéricos foram mantidos fora do target, exceto acordos pós-sentença ou em execução.

## 3. Indicadores principais

- Taxa de vitória do banco.
- Taxa de perda do banco.
- Valor total ajuizado.
- Valor total dos processos perdidos.
- Share do valor perdido.
- Valor médio perdido.
- Assuntos e produtos com maior perda histórica.
- Fases, comarcas, escritórios e órgãos com maior concentração de perda.

## 4. Entregáveis principais

- `executive_summary_eda_valorajuizado_target.csv`
- `target_distribution_banco_ganhou.csv`
- `valorajuizado_by_target_banco_ganhou.csv`
- `prioridade_financeira_target_by_nm_assunto.csv`
- `target_valorajuizado_by_produto_acao_assunto.csv`
- `curva_ganho_financeiro_valor_perdido_summary.csv`
- `ranking_processos_encerrados_por_valor_perdido.csv`

## 5. Próximo passo

Depois de validar a regra de target com o jurídico, o próximo notebook pode ser:

```text
06_feature_engineering_modelo_target_banco_ganhou.ipynb
```

Cuidados importantes para modelagem:

- `Motivo_Encerramento` é target/leakage e não deve entrar como feature.
- `Situacao` também não deve entrar, porque o modelo futuro não pode depender do encerramento.
- `Dataencerramento` não deve entrar como feature preditiva.
- Campos DeepLegal de resultado, sentença, condenação e acordo podem gerar leakage.
- O split deve ser temporal.